# Changepoint Detection
This short notebook shows how to pull changepoint detection records from the Atomscale API using `client.get_changepoints()`. Changepoints are time-bounded regions where an automated detector flagged a meaningful shift in one of the analyzed properties of a data item.

### Install package

In [ ]:
#!pip install atomscale

### API Client Setup

In [ ]:
from atomscale import Client
from atomscale.results import ChangepointResult

api_key = "YOUR_API_KEY_HERE"
client = Client(api_key=api_key)

# Use a data ID from your catalogue. Search or grab one from the web interface.
data_id = "YOUR_DATA_ID_HERE"

### Fetching Changepoints
Call `get_changepoints` with one or more `data_ids` to get back a pandas `DataFrame` of detected changepoints. With default arguments it returns the most recent intensity-profile detection run, filtered to the critical severity tier — a good starting point for most users.

In [ ]:
changepoints = client.get_changepoints(data_ids=data_id)
changepoints

Each row is a single changepoint. The columns are:
```
['id', 'data_id', 'data_modality', 'property_name', 'severity', 'score',
 'window_start_elapsed', 'window_end_elapsed', 'detection_method', 'detail', 'label']
```
- `score` is a normalized magnitude in `[0, 1]` — higher means stronger signal.
- `window_start_elapsed` / `window_end_elapsed` are seconds from the start of the source time series, so you can line changepoints up against timeseries data pulled via `client.get()`.
- `detail` holds method-specific metadata (e.g. RMS error, comparison window size).
- `label` is the applied category label if one has been set, otherwise `None`.

### Filtering Options
The defaults keep the view focused, but all three filter arguments can be adjusted:

- `detection_method` — `"forecasting"`, `"clustering"`, `"intensity_profile"`, or `None` for all methods. Defaults to `"intensity_profile"`.
- `severity` — `"info"`, `"warning"`, `"critical"`, or `None` for all levels. Defaults to `"critical"`.
- `latest_only` — when `True` (default), only anomalies from the most recently completed run per `(data_id, detection_method)` are returned. Set to `False` to include every historical run.

In [ ]:
# Everything the detector flagged, no filters applied.
all_changepoints = client.get_changepoints(
    data_ids=data_id,
    detection_method=None,
    severity=None,
)
all_changepoints[["detection_method", "severity", "score", "property_name"]]

In [ ]:
# All historical runs for this data_id, not just the latest.
history = client.get_changepoints(
    data_ids=data_id,
    detection_method=None,
    severity=None,
    latest_only=False,
)
print(f"latest run only: {len(all_changepoints)} rows")
print(f"all historical runs: {len(history)} rows")

### Result Objects
Pass `as_dataframe=False` to get a list of `ChangepointResult` objects instead of a DataFrame. Useful when you want to iterate and work with the records programmatically.

In [ ]:
results = client.get_changepoints(data_ids=data_id, as_dataframe=False)

for cp in results[:5]:
    window = f"[{cp.window_start_elapsed:.1f}s \u2192 {cp.window_end_elapsed:.1f}s]"
    print(f"{cp.severity:>8}  score={cp.score:.3f}  {window}  {cp.property_name}")

### Batch Over Multiple Data IDs
`data_ids` accepts a list; requests are chunked under the hood so you can pass an arbitrary number of IDs in a single call.

In [ ]:
# Use a search to pull a batch of data IDs, then fetch changepoints across all of them.
search_results = client.search(data_type="rheed_stationary", status="success")
data_ids = search_results["Data ID"].to_list()

batch = client.get_changepoints(data_ids=data_ids)
batch.groupby("data_id").size().sort_values(ascending=False).head()

For more information on other data from the API or other example use see notebooks in the code repository (https://github.com/atomic-data-sciences/api-client) and the documentation (https://atomic-data-sciences.github.io/api-client/)